In [1]:
import sys
from dataclasses import dataclass, field
from itertools import product
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from IPython.display import display
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.ticker import FixedLocator, NullLocator
from tqdm.auto import tqdm

project_root = Path.cwd()
if not (project_root / "tasks").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    import scienceplots  # noqa: F401
except ModuleNotFoundError:
    pass
else:
    plt.style.use("science")

from network_creation.plotting.plot_pair import plot_pair
from network_creation.plotting.reference_values import (
    get_observer_reference_style,
    get_sigmoid_fit_reference_values,
    positive_reference_norm,
)
from network_creation.stats.sigmoid_fit import fit_zero_start_sigmoids
from tasks.policies import load_policy_results

In [2]:
%matplotlib widget


In [3]:
LAST_STEP = "Last step"
SIGMOID_VALUES = {
    "Final plateau (A)": ("A", "estimated final plateau"),
    "Approach speed (k)": ("k", "estimated approach speed"),
    "Time shift (t0)": ("t0", "estimated time shift"),
}


@dataclass(frozen=True)
class PlotRequest:
    observable: str | None = None
    n_agents: int | None = None
    proportion: float | None = None
    value: str | None = None


@dataclass(frozen=True)
class PlaygroundConfig:
    result_store: str | Path | None = None
    final_observables: tuple[str, ...] | None = None
    fit_observables: tuple[str, ...] = (
        "centered_mean_welfare.mean",
        "central_point_of_dominance.mean",
    )
    fit_sigmoids: bool = False
    plot: PlotRequest = field(default_factory=PlotRequest)
    method: str = "gaussian"
    resolution: int = 50
    point_size: float = 3
    pair_atol: float = 1e-5

In [4]:
@dataclass(frozen=True)
class ResultContext:
    project_root: Path
    result_store_path: Path
    ds: xr.Dataset
    proportion_dim: str
    theta_labels: tuple[str, ...]
    observable_labels: tuple[str, ...]
    steps: np.ndarray


@dataclass(frozen=True)
class PlotData:
    observable: str
    n_agents: int
    proportion: float
    value: str
    theta: np.ndarray
    y: np.ndarray
    references: dict[str, float]
    norm: Normalize
    title: str
    colorbar_label: str
    n_pair_axes: int


def resolve_result_store(root, value):
    if value is not None:
        path = Path(value).expanduser()
        return path if path.is_absolute() else root / path
    candidates = sorted(
        root.glob("artifacts/**/*.zarr"), key=lambda p: p.stat().st_mtime, reverse=True
    )
    if not candidates:
        raise FileNotFoundError("No .zarr result stores found under artifacts/")
    return candidates[0]


def load_context(config, root):
    ds = load_policy_results(resolve_result_store(root, config.result_store))
    proportion_dims = tuple(dim for dim in ds["x"].dims if dim.endswith("_proportion"))
    if len(proportion_dims) != 1:
        raise ValueError(
            f"Expected one policy-proportion dimension, found {proportion_dims}"
        )
    return ResultContext(
        project_root=root,
        result_store_path=Path(ds.attrs["result_store_path"]),
        ds=ds,
        proportion_dim=proportion_dims[0],
        theta_labels=tuple(str(v) for v in ds.coords["theta_parameter"].values),
        observable_labels=tuple(str(v) for v in ds.coords["observable"].values),
        steps=np.asarray(ds.coords["step"].values, dtype=float),
    )


def configured_observables(ctx, requested, *, strict):
    if requested is None:
        return ctx.observable_labels
    selected = tuple(str(value) for value in requested)
    missing = tuple(value for value in selected if value not in ctx.observable_labels)
    if missing and strict:
        raise ValueError(
            f"Unknown observables: {missing}. Available: {ctx.observable_labels}"
        )
    if missing:
        print(f"Skipping missing observables: {missing}")
    return tuple(value for value in selected if value in ctx.observable_labels)


def observer_value_labels(config):
    if config.fit_sigmoids:
        return (LAST_STEP, *SIGMOID_VALUES)
    return (LAST_STEP,)


def empty_observer_values(ctx, config):
    value_labels = observer_value_labels(config)
    dims = ("observable", "n_agents", ctx.proportion_dim, "value", "parameter_row")
    shape = (
        ctx.ds.sizes["observable"],
        ctx.ds.sizes["n_agents"],
        ctx.ds.sizes[ctx.proportion_dim],
        len(value_labels),
        ctx.ds.sizes["parameter_row"],
    )
    coords = {dim: ctx.ds.coords[dim] for dim in dims if dim != "value"}
    coords["value"] = ("value", list(value_labels))
    return xr.DataArray(
        np.full(shape, np.nan, dtype=np.float32),
        dims=dims,
        coords=coords,
        name="observer_values",
    )


def build_last_step_values(ctx, config, values):
    observables = configured_observables(ctx, config.final_observables, strict=True)
    tasks = list(
        product(
            observables,
            ctx.ds.coords["n_agents"].values,
            ctx.ds.coords[ctx.proportion_dim].values,
        )
    )
    final = ctx.ds["x"].isel(step=-1)
    for observable, n_agents, proportion in tqdm(
        tasks, desc="Last-step values", unit="map"
    ):
        values.loc[
            dict(
                observable=observable,
                n_agents=n_agents,
                **{ctx.proportion_dim: proportion},
                value=LAST_STEP,
            )
        ] = final.sel(
            observable=observable, n_agents=n_agents, **{ctx.proportion_dim: proportion}
        ).to_numpy()


def build_sigmoid_values(ctx, config, values):
    observables = configured_observables(ctx, config.fit_observables, strict=False)
    tasks = list(
        product(
            observables,
            ctx.ds.coords["n_agents"].values,
            ctx.ds.coords[ctx.proportion_dim].values,
        )
    )
    for observable, n_agents, proportion in tqdm(
        tasks, desc="Sigmoid fits", unit="fit"
    ):
        selector = {
            "observable": observable,
            "n_agents": n_agents,
            ctx.proportion_dim: proportion,
        }
        Y = ctx.ds["x"].sel(selector).transpose("parameter_row", "step").to_numpy()
        params, _ = fit_zero_start_sigmoids(Y, t=ctx.steps)
        params = np.asarray(params, dtype=np.float32)
        for col, value_label in enumerate(SIGMOID_VALUES):
            values.loc[
                dict(
                    observable=observable,
                    n_agents=n_agents,
                    **{ctx.proportion_dim: proportion},
                    value=value_label,
                )
            ] = params[:, col]


def build_observer_values(ctx, config):
    values = empty_observer_values(ctx, config)
    build_last_step_values(ctx, config, values)
    if config.fit_sigmoids:
        build_sigmoid_values(ctx, config, values)
    return values

In [5]:
def pair_axes_count(theta, atol):
    return int(np.count_nonzero(np.var(theta, axis=0) > atol) - 1)


def finite_mask(theta, y):
    return np.isfinite(theta).all(axis=1) & np.isfinite(y)


def finite_reference_values(references):
    values = references.values() if hasattr(references, "values") else references
    return [
        float(value)
        for value in values
        if value is not None and np.isfinite(float(value))
    ]


def resolved_norm(values, references=(), norm=None):
    finite = np.asarray(
        [
            *np.asarray(values, dtype=float).ravel(),
            *finite_reference_values(references),
        ],
        dtype=float,
    )
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return norm if norm is not None else Normalize(vmin=0.0, vmax=1.0)
    vmin = float(finite.min())
    vmax = float(finite.max())
    if norm is None:
        return Normalize(vmin=vmin, vmax=vmax)
    if norm.vmin is not None and np.isfinite(norm.vmin):
        vmin = min(vmin, float(norm.vmin))
    if norm.vmax is not None and np.isfinite(norm.vmax):
        vmax = max(vmax, float(norm.vmax))
    if isinstance(norm, TwoSlopeNorm):
        return TwoSlopeNorm(vmin=vmin, vcenter=norm.vcenter, vmax=vmax)
    return Normalize(vmin=vmin, vmax=vmax, clip=norm.clip)


def candidate_keys(ctx, values, request):
    for observable, n_agents, proportion, value in product(
        values.coords["observable"].values,
        values.coords["n_agents"].values,
        values.coords[ctx.proportion_dim].values,
        values.coords["value"].values,
    ):
        if request.observable is not None and str(observable) != request.observable:
            continue
        if request.n_agents is not None and int(n_agents) != int(request.n_agents):
            continue
        if request.proportion is not None and float(proportion) != float(
            request.proportion
        ):
            continue
        if request.value is not None and str(value) != request.value:
            continue
        yield str(observable), int(n_agents), float(proportion), str(value)


def references_and_norm(
    ctx,
    observable,
    n_agents,
    value,
    theta,
    y,
    *,
    norm_values=None,
    norm_references=None,
):
    values_for_norm = y if norm_values is None else norm_values
    if value == LAST_STEP:
        references, _ = get_observer_reference_style(
            observable,
            y,
            n_agents=n_agents,
            theta_labels=ctx.theta_labels,
            theta_values=theta,
        )
        references_for_norm = references if norm_references is None else norm_references
        return (
            references,
            resolved_norm(values_for_norm, references_for_norm),
            observable,
        )

    reference_key, label = SIGMOID_VALUES[value]
    sigmoid_refs = get_sigmoid_fit_reference_values(
        observable,
        n_agents=n_agents,
    )
    references = sigmoid_refs.get(reference_key, {})
    references_for_norm = references if norm_references is None else norm_references
    if value == "Final plateau (A)":
        _, norm = get_observer_reference_style(
            observable,
            y,
            n_agents=n_agents,
            theta_labels=ctx.theta_labels,
            theta_values=theta,
        )
        return (
            references,
            resolved_norm(values_for_norm, references_for_norm, norm),
            label,
        )
    return (
        references,
        positive_reference_norm(values_for_norm, references_for_norm),
        label,
    )


def shared_n_agent_colorbar_inputs(ctx, values, observable, proportion, value):
    theta_data = ctx.ds["theta"]
    y_values = []
    references = {}
    base_selector = {
        "observable": observable,
        ctx.proportion_dim: proportion,
        "value": value,
    }
    selected_values = values.sel(base_selector)
    for n_agents in selected_values.coords["n_agents"].values:
        n_agents = int(n_agents)
        y = selected_values.sel(n_agents=n_agents).to_numpy()
        theta = theta_data.sel(
            n_agents=n_agents, **{ctx.proportion_dim: proportion}
        ).to_numpy()
        mask = finite_mask(theta, y)
        if not mask.any():
            continue
        y_values.append(y[mask])
        n_references, _, _ = references_and_norm(
            ctx, observable, n_agents, value, theta[mask], y[mask]
        )
        references.update(
            {f"{n_agents}:{name}": ref for name, ref in n_references.items()}
        )
    if not y_values:
        return np.asarray([], dtype=float), references
    return np.concatenate(y_values), references


def select_plot_data(ctx, values, request, config, *, share_colorbar_n_agents=False):
    theta_data = ctx.ds["theta"]
    for observable, n_agents, proportion, value in candidate_keys(ctx, values, request):
        selector = {
            "observable": observable,
            "n_agents": n_agents,
            ctx.proportion_dim: proportion,
            "value": value,
        }
        y = values.sel(selector).to_numpy()
        theta = theta_data.sel(
            n_agents=n_agents, **{ctx.proportion_dim: proportion}
        ).to_numpy()
        mask = finite_mask(theta, y)
        if not mask.any():
            continue
        n_pair_axes = pair_axes_count(theta[mask], config.pair_atol)
        if n_pair_axes < 1:
            continue
        norm_values = None
        norm_references = None
        if share_colorbar_n_agents:
            norm_values, norm_references = shared_n_agent_colorbar_inputs(
                ctx, values, observable, proportion, value
            )
        references, norm, colorbar_label = references_and_norm(
            ctx,
            observable,
            n_agents,
            value,
            theta[mask],
            y[mask],
            norm_values=norm_values,
            norm_references=norm_references,
        )
        return PlotData(
            observable=observable,
            n_agents=n_agents,
            proportion=proportion,
            value=value,
            theta=theta[mask],
            y=y[mask],
            references=references,
            norm=norm,
            title=value,
            colorbar_label=colorbar_label,
            n_pair_axes=n_pair_axes,
        )
    raise ValueError(f"No finite observer values match {request}")

In [6]:
def plot_figsize(base, *, n_axes):
    side = 5.0 if base is None else float(base)
    if n_axes == 1:
        return (side, side * 1.2)
    return (side * n_axes, side * n_axes)


def set_figure_layout(fig, layout):
    if hasattr(fig, "set_layout_engine"):
        fig.set_layout_engine(layout)


def plot_observer_map(
    ctx,
    data,
    config,
    cmap,
    *,
    display_plot=True,
    figsize=None,
    minor_ticks=True,
    fig=None,
):
    n = data.n_pair_axes
    target_figsize = plot_figsize(figsize, n_axes=n)
    if fig is None:
        with plt.ioff():
            fig = plt.figure(figsize=target_figsize)
    else:
        fig.clear()
        fig.set_size_inches(*target_figsize, forward=True)

    if n == 1:
        set_figure_layout(fig, None)
        grid = fig.add_gridspec(1, 1, left=0.08, right=0.72, bottom=0.12, top=0.90)
        axes = np.empty((1, 1), dtype=object)
        axes[0, 0] = fig.add_subplot(grid[0, 0])
        colorbar_ax = fig.add_axes([0.82, 0.12, 0.025, 0.78])
    else:
        set_figure_layout(fig, "constrained")
        axes = fig.subplots(n, n, squeeze=False)

    plot_pair(
        X=data.theta,
        y=data.y,
        labels=ctx.theta_labels,
        atol=config.pair_atol,
        cmap=cmap,
        method=config.method,
        resolution=config.resolution,
        point_size=config.point_size,
        norm=data.norm,
        fig=fig,
        axes=axes,
        ordered_discrete_keys=[
            key for key in ("visibility_distance",) if key in ctx.theta_labels
        ],
    )

    if n > 1:
        colorbar_slot_ax = axes[0, -1]
        colorbar_slot_ax.clear()
        colorbar_slot_ax.axis("off")
        colorbar_ax = colorbar_slot_ax.inset_axes([0.42, 0.0, 0.08, 1.0])

    colorbar = fig.colorbar(
        plt.cm.ScalarMappable(cmap=cmap, norm=data.norm),
        cax=colorbar_ax,
        format="%.2f",
    )
    colorbar.set_label(data.colorbar_label)

    references = sorted(
        (
            (str(name), float(value))
            for name, value in data.references.items()
            if np.isfinite(value)
        ),
        key=lambda item: item[1],
    )
    reference_labels = colorbar_reference_labels(references, data.norm)
    colorbar.set_ticks([value for _, value in reference_labels])
    if minor_ticks:
        colorbar_ax.yaxis.set_minor_locator(
            FixedLocator(colorbar_minor_tick_values(data.norm))
        )
    else:
        colorbar_ax.yaxis.set_minor_locator(NullLocator())
    colorbar_ax.tick_params(
        axis="y",
        which="major",
        left=True,
        right=True,
        labelleft=False,
        labelright=False,
        length=4,
    )
    colorbar_ax.tick_params(
        axis="y",
        which="minor",
        left=minor_ticks,
        right=minor_ticks,
        labelleft=False,
        labelright=False,
        length=2 if minor_ticks else 0,
    )
    colorbar_ax.yaxis.set_label_position("right")
    colorbar_ax.yaxis.labelpad = 34

    for name, value in reference_labels:
        va = colorbar_reference_label_vertical_alignment(value, data.norm)
        name_text = colorbar_ax.text(
            -0.35,
            value,
            name,
            transform=colorbar_ax.get_yaxis_transform(),
            ha="right",
            va=va,
            fontsize="small",
            clip_on=False,
        )
        value_text = colorbar_ax.text(
            1.35,
            value,
            f"{value:.2f}",
            transform=colorbar_ax.get_yaxis_transform(),
            ha="left",
            va=va,
            fontsize="small",
            clip_on=False,
        )
        name_text.set_in_layout(False)
        value_text.set_in_layout(False)

    fig.suptitle(
        "\n".join(
            [
                str(data.title),
                f"n_agents={data.n_agents}, "
                f"{ctx.proportion_dim}={data.proportion:.3g}, "
                f"{data.observable}",
            ]
        )
    )
    if display_plot:
        display(fig.canvas)
    return fig, axes, colorbar_ax


def colorbar_reference_labels(references, norm):
    labels = list(references)
    value = norm.vmin
    if value is not None and np.isfinite(value):
        value = float(value)
        if not any(np.isclose(value, existing_value) for _, existing_value in labels):
            labels.append(("", value))
    return sorted(labels, key=lambda item: item[1])


def colorbar_reference_label_vertical_alignment(value, norm):
    y = float(norm(value))
    if y <= 0.02:
        return "bottom"
    if y >= 0.98:
        return "top"
    return "center"


def colorbar_minor_tick_values(norm, *, count=10):
    if norm.vmin is None or norm.vmax is None:
        return []
    vmin = float(norm.vmin)
    vmax = float(norm.vmax)
    if not np.isfinite(vmin) or not np.isfinite(vmax):
        return []
    if np.isclose(vmin, vmax):
        return [vmin]
    return np.linspace(vmin, vmax, count)

In [7]:
def available_plot_keys(ctx, values):
    has_data = values.notnull().any("parameter_row")
    keys = []
    for observable, n_agents, proportion, value in product(
        values.coords["observable"].values,
        values.coords["n_agents"].values,
        values.coords[ctx.proportion_dim].values,
        values.coords["value"].values,
    ):
        selector = {
            "observable": observable,
            "n_agents": n_agents,
            ctx.proportion_dim: proportion,
            "value": value,
        }
        if bool(has_data.sel(selector).item()):
            keys.append((str(observable), int(n_agents), float(proportion), str(value)))
    return keys


def proportion_label(name):
    return f"{name.removesuffix('_proportion').replace('_', ' ')} share"


def labeled(title, widget):
    return widgets.VBox([widgets.HTML(f"<b>{title}</b>"), widget])


def make_plot_widget(
    ctx,
    values,
    config,
    cmap,
    *,
    figsize=None,
    minor_ticks=True,
    share_colorbar_n_agents=True,
):
    keys = available_plot_keys(ctx, values)
    if not keys:
        raise ValueError("No observer values are available to plot.")

    def matches(observable=None, n_agents=None, proportion=None):
        return [
            key
            for key in keys
            if (observable is None or key[0] == observable)
            and (n_agents is None or key[1] == n_agents)
            and (proportion is None or key[2] == proportion)
        ]

    def unique(index, rows):
        return list(dict.fromkeys(row[index] for row in rows))

    observer = widgets.Select(
        rows=8, layout=widgets.Layout(width="320px", height="170px")
    )
    n_agents = widgets.Select(
        rows=5, layout=widgets.Layout(width="90px", height="120px")
    )
    proportion = widgets.Select(
        rows=5, layout=widgets.Layout(width="170px", height="120px")
    )
    value = widgets.Select(rows=5, layout=widgets.Layout(width="230px", height="120px"))
    plot_area = widgets.Box()

    observer.options = unique(0, keys)

    state = {"updating": False, "fig": None}

    def set_options(control, options):
        control.options = options
        valid_values = [
            item[1] if isinstance(item, tuple) else item for item in options
        ]
        if control.value not in valid_values:
            control.value = valid_values[0]

    def refresh_options(*_):
        if state["updating"]:
            return

        state["updating"] = True
        try:
            n_opts = unique(1, matches(observer.value))
            set_options(n_agents, n_opts)

            p_opts = unique(2, matches(observer.value, n_agents.value))
            set_options(proportion, [(f"{p:.2g}", p) for p in p_opts])

            v_opts = unique(
                3, matches(observer.value, n_agents.value, proportion.value)
            )
            set_options(value, v_opts)
        finally:
            state["updating"] = False

        draw()

    def draw(*_):
        if state["updating"]:
            return

        data = select_plot_data(
            ctx,
            values,
            PlotRequest(observer.value, n_agents.value, proportion.value, value.value),
            config,
            share_colorbar_n_agents=share_colorbar_n_agents,
        )
        fig, _, _ = plot_observer_map(
            ctx,
            data,
            config,
            display_plot=False,
            figsize=figsize,
            minor_ticks=minor_ticks,
            fig=state["fig"],
            cmap=cmap,
        )
        if state["fig"] is None:
            state["fig"] = fig
            plot_area.children = (fig.canvas,)
        else:
            fig.canvas.draw_idle()

    for control in (observer, n_agents, proportion):
        control.observe(refresh_options, names="value")

    value.observe(draw, names="value")

    refresh_options()

    controls = widgets.HBox(
        [
            labeled("Observer", observer),
            labeled("n agents", n_agents),
            labeled(proportion_label(ctx.proportion_dim), proportion),
            labeled("Value", value),
        ],
        layout=widgets.Layout(
            gap="18px", align_items="flex-start", flex_flow="row wrap"
        ),
    )
    return widgets.VBox([controls, plot_area])

In [17]:
config = PlaygroundConfig(
    result_store=project_root
    / "artifacts"
    / "simulation_runs"
    / "run-20260507T191039.zarr",
    method="gaussian",
    fit_sigmoids=False,
    resolution=50,
    point_size=4,
)

ctx = load_context(config, project_root)
observer_values = build_observer_values(ctx, config)

print(f"Loaded {ctx.result_store_path.relative_to(ctx.project_root)}")

Last-step values:   0%|          | 0/132 [00:00<?, ?map/s]

Loaded artifacts/simulation_runs/run-20260507T191039.zarr


In [20]:
display(
    make_plot_widget(
        ctx,
        observer_values,
        config,
        figsize=4,
        minor_ticks=False,
        share_colorbar_n_agents=True,
        cmap="jet",
    )
)

In [19]:
def safe_filename_part(value):
    text = str(value).strip().lower()
    text = "".join(ch if ch.isalnum() else "-" for ch in text)
    return "-".join(part for part in text.split("-") if part) or "value"


def plot_output_path(plot_dir, ctx, key):
    observable, n_agents, proportion, value = key
    proportion_name = ctx.proportion_dim.removesuffix("_proportion")
    filename = "__".join(
        [
            safe_filename_part(observable),
            f"n-agents-{int(n_agents)}",
            f"{safe_filename_part(proportion_name)}-{safe_filename_part(f'{float(proportion):.6g}')}",
            safe_filename_part(value),
        ]
    )
    return plot_dir / f"{filename}.png"


plot_dir = project_root / "artifacts" / "plots" / ctx.result_store_path.stem
plot_dir.mkdir(parents=True, exist_ok=True)

saved_paths = []
skipped = []

for key in tqdm(available_plot_keys(ctx, observer_values), desc="Saving plots", unit="plot"):
    observable, n_agents, proportion, value = key
    fig = None
    try:
        data = select_plot_data(
            ctx,
            observer_values,
            PlotRequest(observable, n_agents, proportion, value),
            config,
            share_colorbar_n_agents=True,
        )
        fig, _, _ = plot_observer_map(
            ctx,
            data,
            config,
            display_plot=False,
            figsize=4,
            minor_ticks=False,
            cmap="jet",
        )
        path = plot_output_path(plot_dir, ctx, key)
        fig.savefig(path, dpi=200, bbox_inches="tight")
        saved_paths.append(path)
    except ValueError as exc:
        skipped.append((key, str(exc)))
    finally:
        if fig is not None:
            plt.close(fig)

print(f"Saved {len(saved_paths)} plots to {plot_dir.relative_to(project_root)}")
if skipped:
    print(f"Skipped {len(skipped)} plot keys without finite pair data.")

Saving plots:   0%|          | 0/132 [00:00<?, ?plot/s]

KeyboardInterrupt: 